# PhantomMap — Colab T4 quickstart

Run every cell top-to-bottom. The full POPE + AMBER sweep on both VLMs takes roughly 10 hours of T4 time, so plan to run it across multiple Colab sessions — every script resumes from its jsonl output file.

**Before running**: in Colab, go to `Runtime -> Change runtime type` and select `T4 GPU`.

In [ ]:
# 1. Clone the project (assumes you pushed it to GitHub).
# Replace the URL with your own repo before running.
!git clone https://github.com/YOUR_GITHUB/phantommap.git /content/phantommap
%cd /content/phantommap

In [ ]:
# 2. Install dependencies.
!pip install -q -r requirements.txt

In [ ]:
# 3. Download data (POPE + AMBER + referenced COCO images).
!python src/download_data.py --out data

In [ ]:
# 4a. Qwen2.5-VL-7B on POPE adversarial (~80 min on T4).
!python src/run_vlm.py --model qwen --split pope_adversarial --out results/qwen_pope_adversarial.jsonl

In [ ]:
# 4b-d. Remaining Qwen splits.
!python src/run_vlm.py --model qwen --split pope_popular --out results/qwen_pope_popular.jsonl
!python src/run_vlm.py --model qwen --split pope_random  --out results/qwen_pope_random.jsonl
!python src/run_vlm.py --model qwen --split amber        --out results/qwen_amber.jsonl

In [ ]:
# 5. LLaVA-NeXT across the same splits (another ~4 hours).
!python src/run_vlm.py --model llava --split pope_adversarial --out results/llava_pope_adversarial.jsonl
!python src/run_vlm.py --model llava --split pope_popular     --out results/llava_pope_popular.jsonl
!python src/run_vlm.py --model llava --split pope_random      --out results/llava_pope_random.jsonl
!python src/run_vlm.py --model llava --split amber            --out results/llava_amber.jsonl

In [ ]:
# 6. Atlas (KDE heatmaps).
!python src/atlas.py \
    --inputs results/*.jsonl \
    --out-fig report/figures/fig3_atlas.pdf \
    --out-stats results/atlas_stats.json

In [ ]:
# 7. Detector for each model separately.
!python src/detector.py \
    --inputs results/qwen_*.jsonl \
    --model-filter "Qwen/Qwen2.5-VL-7B-Instruct" \
    --out results/detector_qwen.json

!python src/detector.py \
    --inputs results/llava_*.jsonl \
    --model-filter "llava-hf/llava-v1.6-mistral-7b-hf" \
    --out results/detector_llava.json

In [ ]:
# 8. All figures (Fig 1 teaser, Fig 3 atlas, Fig 4 ROC, Fig 5 failures).
!python src/make_fig2_method.py --out report/figures/fig2_method.pdf
!python src/make_figures.py \
    --predictions results/*.jsonl \
    --detector-metrics results/detector_qwen.json results/detector_llava.json

In [ ]:
# 9. (Stretch) HITs cross-reference on Qwen. Heavy memory; drop if OOM.
!python src/hits_cross_ref.py \
    --predictions results/qwen_pope_adversarial.jsonl \
    --out results/hits_crossref.jsonl \
    --limit 200

In [ ]:
# 10. Peek at the headline numbers — these are what we'll paste into the LaTeX tables.
import json, glob
from pathlib import Path

print('=== Detector ===')
for p in sorted(glob.glob('results/detector_*.json')):
    m = json.load(open(p))
    print(f"{p}: AUROC full={m['auroc_full']:.4f}  logit-only={m['auroc_logit_only']:.4f}  n_test={m['n_test']}")

print('\n=== Atlas ===')
print(open('results/atlas_stats.json').read())